# 01 · Chat Completions basics

Request anatomy, prompt separation, and a first end-to-end use case: summarizing a web page.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # repo root, for `toolkit`

In [ ]:
from toolkit import PROVIDERS, chat, fetch_page
from toolkit.notebook import show

PROVIDER = "ollama"
PROVIDERS[PROVIDER]

## Request shape

A request is an ordered list of role-tagged messages:

| role | used for |
|---|---|
| `system` | task definition, constraints, tone, output format |
| `user` | per-call input |
| `assistant` | prior model turns (conversation state) |

In [ ]:
chat(
    [
        {"role": "system", "content": "Answer in exactly one sentence."},
        {"role": "user", "content": "Why is the sky blue?"},
    ],
    provider=PROVIDER,
)

## Page summarization

Static instructions go in the system prompt; the scraped page goes in the user message.
Page text is truncated (`Page.as_prompt`) to keep token usage predictable.

In [ ]:
SUMMARY_SYSTEM = """
You summarize websites for someone deciding whether to visit them.
Ignore navigation and boilerplate. Highlight any news or announcements.
Respond in markdown without wrapping it in a code block.
"""


def summary_messages(page):
    return [
        {"role": "system", "content": SUMMARY_SYSTEM},
        {"role": "user", "content": f"Website: {page.url}\n\n{page.as_prompt()}"},
    ]


def summarize(url, provider=PROVIDER):
    return chat(summary_messages(fetch_page(url)), provider=provider)


show(summarize("https://simonwillison.net"))

### Provider comparison

In [ ]:
for provider in ["ollama", "openai"]:
    print(f"\n----- {provider} -----")
    try:
        print(summarize("https://www.anthropic.com", provider=provider))
    except Exception as exc:
        print(f"skipped: {exc}")

Observations:
- Output quality depends on the extracted context. JS-rendered sites return almost no text
  with `requests` + BeautifulSoup, and the summary degrades accordingly (a headless browser is needed there).
- llama3.2 (3B) handles summarization acceptably; the frontier model is noticeably more precise.

## Same pattern, different task: subject-line generation

In [ ]:
SUBJECT_SYSTEM = """
You write email subject lines. Given an email body, suggest 3 subject lines:
one direct, one friendly, one urgent. Max 8 words each. Return a numbered list only.
"""

email = """
Hi team, the staging server will be down on Friday from 2pm to 4pm for a database upgrade.
Please merge anything you need tested before 1pm. Production is not affected.
"""

print(
    chat(
        [{"role": "system", "content": SUBJECT_SYSTEM}, {"role": "user", "content": email}],
        provider=PROVIDER,
    )
)